# 17 — tqdm and Gradio: progress bars and a web app around any function

After this notebook you can put an honest progress bar on any loop (items, bytes, pandas, thread pools), explain exactly where its ETA comes from and why it lags, keep bars out of CI logs, then wrap a Python function in a **Gradio** web app, call it from code, stream results, show your tqdm bars in the browser, and prove with a stopwatch why a Gradio app gets slow with five users — including the trap that silently undoes the fix.

**Time:** ~30 min · **Runs:** offline on CPU in about 30 s. Gradio starts real servers on `127.0.0.1` and shuts them down again · **Versions:** tqdm 4.70, Gradio 6.29 · **Course pages:** Python Basics → Package deep dives → *tqdm* and *Gradio*

## Why this matters in interviews

- Every ML pipeline prints tqdm bars, and "why did it say 2 minutes and take 40?" is a real question about moving averages, not about the library.
- Gradio is how most model demos and Hugging Face Spaces are built. Interviewers ask what it is *for*, how it compares with Streamlit and FastAPI, and why a demo falls over when a few people use it at once.
- The two meet in one line — `gr.Progress(track_tqdm=True)` — which is a nice thing to know when someone asks how you would show progress for a long model call.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `tl33` | How does tqdm work, and where does its ETA come from? |
| `tl34` | How do you use progress bars without breaking logs, notebooks or multiprocessing? |
| `tl35` | What is Gradio, and how does it turn a function into an app? |
| `tl36` | Why does a Gradio app get slow with a few users, and how do you fix it? |
| `tl37` | How do you show progress and stream output in a Gradio app? |
| `tl38` | When is Gradio the right tool, and when should you not ship it? |
| `tl07` | Compare FastAPI and Streamlit. |

In [ ]:
import os
os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"      # no telemetry calls from this notebook

import io
import logging
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tqdm as tqdm_pkg
from tqdm import tqdm, trange
from tqdm.std import EMA

logging.getLogger("httpx").setLevel(logging.WARNING)
print("tqdm", tqdm_pkg.__version__)

## 1. tqdm in one line

**In plain English:** wrap the thing you loop over. tqdm counts items as they pass, and a few times a second rewrites one line with the count, the speed and the time left.

**Technically:** `tqdm(iterable)` returns an iterator. Each `next()` bumps a counter; the line is redrawn only when at least `mininterval` (0.1 s) has passed. Here the bar writes into a string buffer so we can read what it printed.

In [ ]:
def last_line(buf):
    # a bar redraws with carriage returns; the final state is the last segment
    return buf.getvalue().replace("\n", "\r").strip("\r").split("\r")[-1]


buf = io.StringIO()
total = 0
for x in tqdm(range(200_000), desc="sum", file=buf, ncols=80):
    total += x
print(last_line(buf))
print("redraws written:", buf.getvalue().count("\r"))

assert total == sum(range(200_000))
assert "200000/200000" in last_line(buf) and last_line(buf).startswith("sum: 100%")

Only a handful of redraws for 200,000 items: that is why wrapping a loop is nearly free. Measure it:

In [ ]:
N = 1_000_000

t0 = time.perf_counter()
for _ in range(N):
    pass
bare = time.perf_counter() - t0

t0 = time.perf_counter()
for _ in tqdm(range(N), file=io.StringIO()):
    pass
wrapped = time.perf_counter() - t0

per_item_ns = (wrapped - bare) / N * 1e9
print(f"bare loop {bare:.3f} s, with tqdm {wrapped:.3f} s -> about {per_item_ns:.0f} ns extra per item")
assert per_item_ns < 1500, "tqdm should add well under a microsecond or two per item"

## 2. Every part of the line

**In plain English:** the line is built by one static function, `tqdm.format_meter(n, total, elapsed, ...)`. Calling it directly shows every piece with no timing noise.

The course page re-implements this function in JavaScript, and its tests check it against **exactly these eight strings**.

In [ ]:
fm = tqdm.format_meter
REAL = [
    (dict(n=420, total=1000, elapsed=12.17, prefix="train", postfix="loss=0.231", ncols=80),
     "train:  42%|████████▊            | 420/1000 [00:12<00:16, 34.51it/s, loss=0.231]"),
    (dict(n=3_500_000, total=10_485_760, elapsed=4.2, prefix="model.safetensors", unit="B", unit_scale=True, unit_divisor=1024, ncols=90),
     "model.safetensors:  33%|██████████                    | 3.34M/10.0M [00:04<00:08, 833kB/s]"),
    (dict(n=37, total=None, elapsed=5, prefix="stream"), "stream: 37it [00:05,  7.40it/s]"),
    (dict(n=1000, total=1000, elapsed=29.0, ncols=70), "100%|█████████████████████████████| 1000/1000 [00:29<00:00, 34.48it/s]"),
    (dict(n=0, total=50, elapsed=0, prefix="embed", ncols=60), "embed:   0%|                         | 0/50 [00:00<?, ?it/s]"),
    (dict(n=3, total=12, elapsed=41.0, prefix="pages", unit="page", ncols=80),
     "pages:  25%|████████▊                          | 3/12 [00:41<02:03, 13.67s/page]"),
    (dict(n=5, total=8, elapsed=3.9, ncols=0), " 62% 5/8 [00:03<00:02,  1.28it/s]"),
    (dict(n=733, total=1200, elapsed=3725.0, prefix="epoch 2", ncols=100, rate=0.42),
     "epoch 2:  61%|█████████████████████████████▉                   | 733/1200 [1:02:05<18:31,  2.38s/it]"),
]
for kwargs, want in REAL:
    got = fm(**kwargs)
    print(got)
    assert got == want, (got, want)

Things worth noticing, each one a parameter:

- **No total → no %, bar or ETA** (`stream: 37it`). A generator has no `len()`.
- **Rate flips to seconds per item** when that reads better: `13.67s/page`, not `0.07page/s`.
- **`unit_scale` + `unit_divisor=1024`** turn 3,500,000 bytes into `3.34M`; the rate stays decimal (`833kB/s`).
- **62.5% prints as `62%`**: Python rounds an exact half to even — a detail the JavaScript copy had to reproduce to match.
- **`remaining = (total − n) / rate`**: `(1000 − 420) / 34.51 = 16.8 s` → `00:16`.

## 3. The API you actually use

In [ ]:
out = io.StringIO()

# generators: pass total= (never list() them just to get a length)
rows = (i * 2 for i in range(50_000))
n = sum(1 for _ in tqdm(rows, total=50_000, desc="rows", file=out))
assert n == 50_000

# bytes: create the bar with a total, then update(n) yourself
data = io.BytesIO(os.urandom(5 * 1024 * 1024))
with tqdm(total=5 * 1024 * 1024, unit="B", unit_scale=True, unit_divisor=1024, desc="copy", file=out) as bar:
    while chunk := data.read(64 * 1024):
        bar.update(len(chunk))
print(last_line(out))
assert bar.n == 5 * 1024 * 1024

# live numbers
out = io.StringIO()
bar = tqdm(range(100), desc="epoch 1", file=out, mininterval=0)
for step in bar:
    bar.set_postfix(loss=f"{1 / (step + 1):.3f}")
bar.close()
print(last_line(out))
assert "loss=0.010" in last_line(out)

# nested: the inner bar vanishes after each outer step
out = io.StringIO()
for epoch in trange(2, desc="epochs", file=out):
    for batch in tqdm(range(3), desc="batches", leave=False, file=out):
        pass

In [ ]:
# pandas: progress_apply is .apply with a bar
from tqdm.auto import tqdm as tqdm_auto        # widget in Jupyter, text elsewhere

tqdm_auto.pandas(desc="clean", file=io.StringIO())
df = pd.DataFrame({"text": [f"  Row {i}  " for i in range(20_000)]})
df["clean"] = df["text"].progress_apply(str.strip)
assert df["clean"].equals(df["text"].apply(str.strip))

# one bar for a whole thread pool, results in input order
from tqdm.contrib.concurrent import thread_map

def slow_square(x):
    time.sleep(0.01)
    return x * x

t0 = time.perf_counter()
squares = thread_map(slow_square, range(64), max_workers=16, desc="pool", file=io.StringIO())
print(f"64 x 10 ms on 16 threads: {time.perf_counter() - t0:.2f} s")
assert squares == [x * x for x in range(64)], "thread_map keeps the input order"

## 4. Where the ETA comes from — and why it lies

**In plain English:** remaining time is "items left ÷ current speed". The question is what "current speed" means. tqdm uses a **moving average**: each redraw it blends the newest measurement into the old estimate. `smoothing` is the weight of the newest one — 0 means "the average over the whole run", 1 means "only the last redraw", and the default is 0.3.

**Technically:** tqdm keeps two exponential moving averages, of items per redraw (`dn`) and seconds per redraw (`dt`), with bias correction so the first few are not dragged towards zero: `rate = EMA(dn) / EMA(dt)`. This is tqdm's own `EMA` class:

In [ ]:
ema = EMA(0.3)
print([round(ema(x), 6) for x in [10, 10, 2, 2, 2]])
assert [round(v, 6) for v in [EMA(0.3)(10)]] == [10.0], "bias correction: the first value is taken as-is"


def simulate(legs, total, smoothing, tick=0.5, noise=0.35, seed=7):
    # a job whose speed depends on which leg it is in, +/- noise, redrawn every `tick` seconds
    rng = np.random.default_rng(seed)
    bounds = np.cumsum([k for k, _ in legs])
    n, t, trace, change = 0.0, 0.0, [], None
    while n < total:
        leg = int(np.searchsorted(bounds, n, side="right"))
        if leg > 0 and change is None:
            change = t
        n = min(total, n + legs[leg][1] * (1 + noise * 2 * (rng.random() - 0.5)) * tick)
        t += tick
        trace.append((t, int(n)))
    end = t
    e_dn, e_dt, prev, rows = EMA(smoothing), EMA(smoothing), 0, []
    for tt, nn in trace[:-1]:
        dn, prev = nn - prev, nn
        if smoothing:
            if dn:
                e_dn(dn); e_dt(tick)
            rate = e_dn() / e_dt()
        else:
            rate = nn / tt
        rows.append((tt, (total - nn) / rate if rate else np.nan, end - tt))
    return pd.DataFrame(rows, columns=["t", "eta", "truth"]), change


def settles(df, change):
    # seconds after the speed change until the ETA is, and stays, within 25% (or 2 s) of the truth
    ok = (df.eta - df.truth).abs() <= np.maximum(0.25 * df.truth, 2)
    after = df[df.t > change]
    for i in range(len(after)):
        if ok[after.index[i]:].all():
            return after.t.iloc[i] - change
    return np.inf


SLOWDOWN = [(600, 50), (400, 10)]           # cached items at 50 it/s, then the network at 10 it/s
res = {s: simulate(SLOWDOWN, 1000, s) for s in (0, 0.3, 1)}
plt.figure(figsize=(8, 3.8))
plt.plot(res[0][0].t, res[0][0].truth, "k--", lw=1.2, label="true remaining")
for s, col in [(0, "#8b5cf6"), (0.3, "#0891b2"), (1, "#d97706")]:
    plt.plot(res[s][0].t, res[s][0].eta.clip(upper=120), color=col, lw=1.6, label=f"ETA, smoothing={s}")
plt.axvline(res[0][1], color="gray", lw=0.8, ls=":")
plt.xlabel("seconds into the job"); plt.ylabel("seconds left"); plt.ylim(0, 80); plt.legend(frameon=False)
plt.title("A job that slows down at 60%"); plt.show()

for s in (0, 0.3, 1):
    df, change = res[s]
    jitter = df.eta.diff().abs().mean()
    print(f"smoothing={s}: settles {settles(df, change):5.1f} s after the slowdown, ETA jumps {jitter:.1f} s per redraw")

S = {s: settles(*res[s]) for s in (0, 0.3, 1)}
assert S[0.3] < S[0] and S[0.3] < S[1], "the default should settle first: 0 remembers the past, 1 chases the noise"

The overall average (violet) remembers the fast phase for the rest of the job. `smoothing=1` (amber) reacts immediately but jumps around so much it never settles either. The default 0.3 is a compromise that happens to work well for most real loops. When a job has distinct phases, the honest fix is not a smoothing value: it is **one bar per phase**, or a total measured in the unit that actually costs time (bytes, tokens) rather than in items.

## 5. Bars in CI logs, and a trap in the fix

**In plain English:** a bar redraws one line with carriage returns. In a log file every redraw becomes a new line. You can turn bars off from outside the code with the environment variable `TQDM_DISABLE=1` (every tqdm argument has a `TQDM_*` twin). The trap: tqdm reads those variables **once, when it is imported**, so setting one inside a script that has already imported tqdm does nothing.

In [ ]:
os.environ["TQDM_DISABLE"] = "1"
print("set after import -> disabled?", tqdm(range(3), file=io.StringIO()).disable)
del os.environ["TQDM_DISABLE"]

probe = "from tqdm import tqdm; import io; print(tqdm(range(3), file=io.StringIO()).disable)"
env = {**os.environ, "TQDM_DISABLE": "1"}
fresh = subprocess.run([sys.executable, "-c", probe], env=env, capture_output=True, text=True).stdout.strip()
print("set before the process starts -> disabled?", fresh)

assert fresh == "True", "TQDM_DISABLE works when it is in the environment before import"
# the in-process result depends only on when tqdm was imported, which is the lesson

Other habits for logs and notebooks: `disable=None` switches the bar off automatically when output is not a terminal; `tqdm.write()` prints without breaking the bar; `logging_redirect_tqdm()` (from `tqdm.contrib.logging`) does the same for loggers; `from tqdm.auto import tqdm` picks the notebook widget in Jupyter.

## 6. Gradio: a web app around a function

**In plain English:** you write a normal Python function. Gradio builds a page with one box per argument and one per return value, and puts a queue and an HTTP API in front of it. We start it on this machine, call it from Python like another program would, and shut it down.

In [ ]:
import gradio as gr
from gradio_client import Client

print("gradio", gr.__version__)


def summarise(text: str, max_words: int) -> str:
    words = text.split()
    return " ".join(words[:max_words]) + ("..." if len(words) > max_words else "")


demo = gr.Interface(
    fn=summarise,
    inputs=[gr.Textbox(lines=6, label="Text"), gr.Slider(5, 100, value=30, step=5, label="Max words")],
    outputs=gr.Textbox(label="Summary"),
    flagging_mode="never",
    api_name="summarise",
)
demo.launch(prevent_thread_lock=True, quiet=True, server_name="127.0.0.1")   # in a script: demo.launch()
print("running at", demo.local_url)

client = Client(demo.local_url, verbose=False)
answer = client.predict("Gradio wraps a Python function in a web page and an API.", 5, api_name="/summarise")
print("predict ->", answer)
print("endpoints:", list(client.view_api(print_info=False, return_format="dict")["named_endpoints"]))
assert answer == "Gradio wraps a Python function..."

The API validates inputs exactly like the page does: the slider's range is enforced on the server, not only in the browser.

In [ ]:
from gradio_client.exceptions import AppError

try:
    client.predict("too short a limit", 3, api_name="/summarise")
    rejected = None
except AppError as e:
    rejected = str(e)
print("rejected:", rejected)
demo.close()
assert rejected and "minimum" in rejected

## 7. Blocks: your own layout, streaming, state and progress

**In plain English:** `gr.Blocks` lets you place components yourself and decide which button runs which function. Three extras matter for model apps:

- **Streaming:** make the function a generator; every `yield` is sent to the page.
- **Per-user state:** `gr.State` is an invisible component. A global variable would be shared by every visitor.
- **Progress:** a default argument `progress=gr.Progress(track_tqdm=True)` mirrors every tqdm bar inside the call to the browser — the bridge between the two halves of this notebook.

In [ ]:
def stream_words(text):
    acc = []
    for w in text.split():
        acc.append(w)
        yield " ".join(acc)                     # each yield reaches the browser


def remember(item, seen):
    seen = seen + [item]                        # return a NEW list: state is replaced, not mutated
    return seen, f"{len(seen)} items this session"


def index_files(n, progress=gr.Progress(track_tqdm=True)):
    for _ in tqdm(range(int(n)), desc="embedding"):   # this bar shows up in the page
        time.sleep(0.01)
    return f"{int(n)} files indexed"


with gr.Blocks() as app:
    text, streamed = gr.Textbox(label="Text"), gr.Textbox(label="Streamed")
    gr.Button("Stream").click(stream_words, text, streamed, api_name="stream")
    seen, status = gr.State([]), gr.Textbox(label="Status")
    gr.Button("Remember").click(remember, [text, seen], [seen, status], api_name="remember")
    n, done = gr.Number(value=20, label="Files"), gr.Textbox(label="Done")
    gr.Button("Index").click(index_files, n, done, api_name="index")

app.launch(prevent_thread_lock=True, quiet=True, server_name="127.0.0.1")
c = Client(app.local_url, verbose=False)

job = c.submit("one two three four", api_name="/stream")
final = job.result()
print("stream outputs:", job.outputs())
print("index ->", c.predict(20, api_name="/index"))
first, second = c.predict("apple", api_name="/remember"), c.predict("pear", api_name="/remember")
other = Client(app.local_url, verbose=False).predict("fig", api_name="/remember")   # a second visitor
print("same client:", first, "|", second, "   new client:", other)   # State itself is not an API output
app.close()

assert final == "one two three four" and len(job.outputs()) >= 2, "a generator must stream several partial results"
assert second.startswith("2 ") and other.startswith("1 "), "state follows the session, not the server"

The same client sees its list grow (1, then 2 items), and a second client starts again at 1: `gr.State` belongs to a **session** — a browser tab, or one `Client` object — not to the server. A global list would have shown the second visitor the first visitor's items.

A `gr.ChatInterface` works the same way: `fn(message, history)`, where `history` is a list of `{"role", "content"}` dicts. Copy just those two keys before sending history to a model API — Gradio may add others (`metadata`, `options`) that strict APIs reject:

In [ ]:
def reply(message, history):
    msgs = [{"role": m["role"], "content": m["content"]} for m in history]
    msgs.append({"role": "user", "content": message})
    words = f"You sent {len(msgs)} messages; the last was {message!r}.".split()
    partial = ""
    for w in words:                              # stream into the chat bubble
        partial = (partial + " " + w).strip()
        yield partial

history = [{"role": "user", "content": "hi", "metadata": None},
           {"role": "assistant", "content": "hello!", "options": []}]
*_, last = reply("what is gradio?", history)
print(last)
assert last == "You sent 3 messages; the last was 'what is gradio?'."
chat = gr.ChatInterface(reply, title="Echo bot")   # built, not launched

## 8. Why it gets slow with five users — measured

**In plain English:** Gradio sends every click through a queue. By default each event runs **one call at a time**, so the fifth user waits for the first four. That protects a GPU from being asked to run two models at once, and it makes an app that only waits on an API feel broken.

We fire 8 calls at once at a function that takes 0.5 s, with plain HTTP (`POST /gradio_api/call/<name>` returns an event id; `GET` on that id streams the result):

1. **default** limit (1)
2. **`concurrency_limit=4`**
3. **the trap:** `concurrency_limit=4`, but another endpoint wraps *the same Python function* with the default limit

In [ ]:
import httpx


def slow_upper(text):
    time.sleep(0.5)
    return text.upper()


def slow_upper_again(text):          # a SECOND function object with the same behaviour
    return slow_upper(text)


with gr.Blocks() as q:
    inp, out = gr.Textbox(), gr.Textbox()
    gr.Button("a").click(slow_upper, inp, out, api_name="default")
    gr.Button("b").click(slow_upper_again, inp, out, api_name="limit4", concurrency_limit=4)
    gr.Button("c").click(slow_upper, inp, out, api_name="shared4", concurrency_limit=4)   # same fn as "default"
q.launch(prevent_thread_lock=True, quiet=True, server_name="127.0.0.1")
base = q.local_url.rstrip("/")


def call(name, text):
    with httpx.Client(trust_env=False, timeout=60) as h:
        event = h.post(f"{base}/gradio_api/call/{name}", json={"data": [text]}).json()["event_id"]
        return h.get(f"{base}/gradio_api/call/{name}/{event}").text.strip().splitlines()[-1]


for name in ("default", "limit4", "shared4"):
    call(name, "warm-up")
T = {}
for name in ("default", "limit4", "shared4"):
    t0 = time.perf_counter()
    with ThreadPoolExecutor(8) as ex:
        results = list(ex.map(lambda i: call(name, f"user {i}"), range(8)))
    T[name] = time.perf_counter() - t0
    print(f"{name:8s}: 8 calls of 0.5 s took {T[name]:.1f} s   ({results[0]})")
q.close()

assert T["default"] > 2.5 * T["limit4"], "the default limit of 1 must serialise the calls"
assert T["shared4"] > 2.5 * T["limit4"], "sharing a function with a limit-1 event must drag limit 4 down to 1"

Case 3 is the surprise. Gradio groups concurrency by `concurrency_id`, and when you do not set one it defaults to **the function's id** — so every event that wraps the same function shares one group, and the group runs at the lowest limit among them. The fixes: a second function object (as `slow_upper_again` does), or an explicit `concurrency_id="…"` per event. And `concurrency_limit=None` is still capped at 40 simultaneous calls for sync functions by `launch(max_threads=40)`.

## 9. Inside a real API

**In plain English:** when the demo has to live next to a production API, mount it into your FastAPI app instead of running two servers.

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

api = FastAPI()

@api.get("/health")
def health():
    return {"ok": True}

api = gr.mount_gradio_app(api, demo, path="/ui")
with TestClient(api) as tc:
    h, ui = tc.get("/health"), tc.get("/ui/")
print("/health ->", h.json(), "| /ui/ ->", ui.status_code, ui.headers.get("content-type"))
assert h.json() == {"ok": True} and ui.status_code == 200

## Try it yourself

**Exercise 1.** Read a 3 MB in-memory file in 100 KB chunks with a byte-accurate bar (binary units), and check the bar's final count equals the file size.

In [ ]:
# Solution — try it yourself first, then run this
blob = io.BytesIO(os.urandom(3 * 1024 * 1024))
size = blob.getbuffer().nbytes
out = io.StringIO()
with tqdm(total=size, unit="B", unit_scale=True, unit_divisor=1024, desc="read", file=out) as bar:
    while chunk := blob.read(100 * 1024):
        bar.update(len(chunk))
print(last_line(out))
assert bar.n == size and "3.00M/3.00M" in last_line(out)

**Exercise 2.** Fix the shared-function trap the other way: keep one function, but give the fast endpoint its own `concurrency_id`. Show that 8 calls now finish in about a second.

In [ ]:
# Solution — try it yourself first, then run this
with gr.Blocks() as fixed:
    inp, out = gr.Textbox(), gr.Textbox()
    gr.Button("a").click(slow_upper, inp, out, api_name="default")
    gr.Button("b").click(slow_upper, inp, out, api_name="fast", concurrency_limit=4, concurrency_id="fast-group")
fixed.launch(prevent_thread_lock=True, quiet=True, server_name="127.0.0.1")
base = fixed.local_url.rstrip("/")
call("fast", "warm-up")
t0 = time.perf_counter()
with ThreadPoolExecutor(8) as ex:
    list(ex.map(lambda i: call("fast", f"user {i}"), range(8)))
elapsed = time.perf_counter() - t0
fixed.close()
print(f"8 calls with their own concurrency group: {elapsed:.1f} s")
assert elapsed < T["default"] / 2.5

**Exercise 3.** Your pipeline has two phases: 200 slow items (8 it/s) then 800 fast ones (80 it/s). Using `simulate` from section 4, how wrong is the overall-average ETA (smoothing=0) at the moment the fast phase starts, compared with the truth?

In [ ]:
# Solution — try it yourself first, then run this
df, change = simulate([(200, 8), (800, 80)], 1000, 0)
row = df[df.t >= change].iloc[0]
print(f"at t={row.t:.1f} s the average says {row.eta:.0f} s left; the truth is {row.truth:.0f} s")
assert row.eta > 3 * row.truth, "after a slow start, the overall average is far too pessimistic"

## Say it in an interview

- **tqdm, 30 seconds:** "tqdm wraps an iterable and redraws one line at most every 0.1 s, so it costs a fraction of a microsecond per item. The ETA is items left divided by a rate that is an exponential moving average of recent redraws — smoothing 0.3 by default, 0 for the overall average — so it lags any change of speed. For bytes I create the bar with a total and call update(n); generators need total=; in CI I disable it with TQDM_DISABLE or disable=None; in notebooks I use tqdm.auto."
- **Gradio, 30 seconds:** "Gradio maps a function to a UI: one component per argument, one per output, one API endpoint per event. Interface for one function, Blocks for custom layout, ChatInterface for chat. Every event goes through a queue with a default concurrency limit of 1 — right for a GPU model, wrong for an I/O-bound function — and events that wrap the same function share one limit. I stream with generators, track progress with gr.Progress(track_tqdm=True), keep per-user data in gr.State, and call it from code with gradio_client."
- **Numbers:** tqdm redraws every 0.1 s, smoothing 0.3; Gradio 6: default concurrency 1, 40 worker threads, share links last up to a week.
- **Traps:** print() inside a tqdm loop; list(gen) for a length; setting TQDM_* after import; global variables in Gradio; share=True on a machine with secrets; two endpoints on one function; Gradio 5 tutorials on Gradio 6 (theme moved to launch(), no Chatbot type=).
- **Follow-ups:** "Gradio or Streamlit?" (functions and events vs a script rerun per interaction). "Would you ship it?" (as a demo or internal tool, yes; as the product, put FastAPI underneath and mount Gradio beside it).

## Next

- [Async/await basics](14_async_await_basics.ipynb) — why an async function lets one Gradio worker hold many waiting calls.
- [FastAPI serving and streaming](../16_production/04_fastapi_serving_and_streaming.ipynb) — the production side of the same app.
- [Async concurrency for LLM calls](../16_production/01_async_concurrency_for_llm_calls.ipynb) — concurrency limits from the client's side.